# Alert 생성기 실험 ② — 묶음이 정답(세탁 시도)과 얼마나 같은가

같은 씨앗(모델 점수 ≥ 의심 기준)을 두 가지 방법으로 묶고, 데이터셋의 **세탁 시도**(Patterns 파일의 BEGIN ~ END 한 덩어리)와 비교한다.

| 방법 | 묶는 규칙 |
|---|---|
| G 생성기 | 서비스 Alert 생성기. 01번 결과(`members.parquet`)를 그대로 읽는다 |
| B 단순 연결 | 씨앗끼리 계좌를 공유하면 같은 묶음. 금액·방향·바쁜 계좌는 보지 않는다. 시간 제한만 바꿔 본다 |

- 정답은 시도마다 **평가 구간 안에 있는 거래**만 쓴다(구간 안 거래가 2건 이상인 시도). 씨앗이 평가 구간에만 있어서 구간 밖 거래는 어느 방법도 묶을 수 없다.
- 첫 버전은 "구간 안에서 시작하고 끝난 시도"만 썼는데, HI-Large 에서는 그런 시도가 103개뿐이고 거의 다 거래 2건짜리라 대표성이 없었다(2026-11-22).
- `EXCLUDE` 의 패턴 거래는 묶음과 정답 양쪽에서 뺀다.
- 01번을 먼저 돌려야 한다. 생성기를 다시 돌리지는 않는다.

## 0. 설정

In [1]:
DATASET = 'HI-Medium'            # 01번을 같은 DATASET 으로 먼저 돌려야 한다
RUN = 'before_after_grouping'            # 01번 결과 폴더
EXCLUDE = ['STACK', 'BIPARTITE']         # 평가에서 빼는 패턴
BASELINES = {'B0 계좌 공유': None, 'B1 계좌 공유 + 7일 이내': 168, 'B2 계좌 공유 + 1일 이내': 24, 'B 4일 이내': 96}   # 이름: 시간 제한(시간)

In [2]:
from pathlib import Path
import json
import duckdb, numpy as np, pandas as pd
import gen_lib as GL

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'parquets').is_dir() or (p / 'datasets').is_dir()), None)
assert ROOT is not None, 'parquets/ 폴더를 못 찾음. 데이터가 있는 GPU 서버의 저장소 안에서 열어야 한다'
BASE = ROOT / 'parquets' / DATASET
RUN_DIR = Path.cwd() / 'results' / DATASET / RUN
OUT = Path.cwd() / 'results' / DATASET / 'grouping_vs_truth'
OUT.mkdir(parents=True, exist_ok=True)
run = json.load(open(RUN_DIR / 'run.json', encoding='utf-8'))
a, b = run['test_window']
print('01번 실행:', run['scores'], '| 기준', run['threshold'], '| 평가 구간', a, '~', b)

01번 실행: ../../parquets/HI-Medium/service_scores/multi280_hgb36_v1_test_scores.parquet | 기준 0.194997844 | 평가 구간 2022-09-08 ~ 2022-09-16


## 1. 01번 결과 읽기
`members.parquet` 에는 씨앗과 Alert 구성 거래가 들어 있다. 계좌와 시각은 거래 파일에서 붙인다.

In [3]:
members = pd.read_parquet(RUN_DIR / 'members.parquet')
con = duckdb.connect()
con.register('member_ids', members[['edge_id']])
tx = con.execute(f"""SELECT t.edge_id, t.ts, t.src, t.dst
                     FROM read_parquet('{BASE / 'transactions.parquet'}') t JOIN member_ids USING (edge_id)""").df()
members = members.merge(tx, on='edge_id', how='left')
assert members.ts.notna().all(), '거래 파일에 없는 edge_id 가 있다'
print(f'거래 {len(members):,} 건 | 씨앗 {int(members.seed_model.sum()):,} | Alert 구성 거래 {int((members.role > 0).sum()):,}')

거래 14,870 건 | 씨앗 13,343 | Alert 구성 거래 11,526


## 2. 정답: 세탁 시도
시도마다 평가 구간 안의 거래 수를 세고, 평가에 쓸 시도를 고른다.

In [4]:
PAT = BASE / 'patterns.parquet'
start, end = pd.Timestamp(a), pd.Timestamp(b) + pd.Timedelta(days=1)
IN = f"ts >= TIMESTAMP '{start}' AND ts < TIMESTAMP '{end}'"
attempts = con.execute(f"""SELECT attempt_id, any_value(pattern_type) AS pattern_type, count(*) AS n_rows,
                                  count(*) FILTER (WHERE {IN}) AS n_in, count(edge_id) FILTER (WHERE {IN}) AS n_in_matched,
                                  min(ts) AS first_ts, max(ts) AS last_ts
                           FROM read_parquet('{PAT}') GROUP BY attempt_id""").df()
attempts['touch'] = attempts.n_in >= 1                                            # 평가 구간에 거래가 있음
attempts['inside'] = (attempts.first_ts >= start) & (attempts.last_ts < end)      # 평가 구간 안에서 시작하고 끝남
attempts['eligible'] = (~attempts.pattern_type.isin(EXCLUDE) & (attempts.n_in >= 2) & (attempts.n_in == attempts.n_in_matched))
attempts[attempts.touch].groupby('pattern_type').agg(걸친_시도=('touch', 'sum'), 구간_안에서_끝남=('inside', 'sum'), 평가_대상=('eligible', 'sum'),
                                                      전체_거래_중앙값=('n_rows', 'median'), 구간_안_거래_중앙값=('n_in', 'median'))

,걸친_시도,구간_안에서_끝남,평가_대상,전체_거래_중앙값,구간_안_거래_중앙값
pattern_type,,,,,
BIPARTITE,241,187,0,5.0,3.0
CYCLE,308,100,236,6.0,3.0
FAN-IN,280,112,209,7.0,3.0
FAN-OUT,268,110,183,6.0,3.0
GATHER-SCATTER,289,45,267,15.0,7.0
RANDOM,254,131,168,5.0,3.0
SCATTER-GATHER,283,88,228,10.0,5.0
STACK,266,96,0,12.0,4.0


In [5]:
# 거래 → 시도. 한 거래가 두 시도에 들어 있으면 번호가 작은 시도로 둔다
attempt_of = con.execute(f"""SELECT edge_id, min(attempt_id) AS attempt_id, arg_min(pattern_type, attempt_id) AS pattern_type, count(*) AS n,
                                    bool_and({IN}) AS in_window
                             FROM read_parquet('{PAT}') WHERE edge_id IS NOT NULL GROUP BY edge_id""").df()
print('두 시도에 동시에 속한 거래:', int((attempt_of.n > 1).sum()))
attempt_of = attempt_of[~attempt_of.pattern_type.isin(EXCLUDE)]
eval_attempts = set(attempts.loc[attempts.eligible, 'attempt_id'])
truth = attempt_of[attempt_of.attempt_id.isin(eval_attempts) & attempt_of.in_window]      # 정답: 평가할 시도의 구간 안 거래
print(f'평가할 시도 {len(eval_attempts):,} 개 | 정답 거래 {len(truth):,} 건')

두 시도에 동시에 속한 거래: 0
평가할 시도 1,291 개 | 정답 거래 8,897 건


In [6]:
# 모델이 애초에 몇 건을 잡았나: 어떤 묶기 방법도 이것보다 잘할 수는 없다(연결 거래로 되찾는 경우 제외)
is_seed = members.seed_model | members.promoted          # 씨앗 = 모델 씨앗 + 보강으로 들어온 씨앗 (01번에서 보강을 켰을 때만 다르다)
seed_ids = set(members.loc[is_seed, 'edge_id'])
t = truth.assign(seed=lambda d: d.edge_id.isin(seed_ids))
by_attempt = t.groupby('attempt_id').agg(size=('seed', 'size'), seeds=('seed', 'sum'))
ceiling = dict(시도_수=len(by_attempt), 모델이_전부_잡은_시도=int((by_attempt.seeds == by_attempt['size']).sum()),
               모델이_2건_이상_잡은_시도=int((by_attempt.seeds >= 2).sum()), 모델이_하나도_못_잡은_시도=int((by_attempt.seeds == 0).sum()))
ceiling

{'시도_수': 1291,
 '모델이_전부_잡은_시도': 1187,
 '모델이_2건_이상_잡은_시도': 1278,
 '모델이_하나도_못_잡은_시도': 1}

## 3. 묶음 만들기
- 묶음은 **거래 2건 이상**만 센다. 생성기는 혼자인 씨앗을 Alert 로 내지 않으므로 단순 연결도 같게 맞춘다.
- 생성기는 제외 패턴의 씨앗도 넣고 돌린 결과에서 그 거래만 뺀다. 단순 연결은 제외 패턴 씨앗을 빼고 잇는다.

In [7]:
keep = ~members.pattern_type.isin(EXCLUDE)

def as_groups(frame, column):
    g = frame[['edge_id', column, 'y']].rename(columns={column: 'group'})
    return g[g.groupby('group')['edge_id'].transform('size') >= 2]

methods = {'G 생성기': as_groups(members[(members.role > 0) & keep], 'alert')}
seeds = members[is_seed & keep].sort_values(['ts', 'edge_id']).reset_index(drop=True)
lonely = {}
for name, hours in BASELINES.items():
    seeds['component'] = GL.link_by_account(seeds, hours)
    methods[name] = as_groups(seeds, 'component')
    lonely[name] = len(seeds) - len(methods[name])
print('씨앗', len(seeds), '| 혼자 남아 묶음이 못 된 씨앗:', lonely)

씨앗 11027 | 혼자 남아 묶음이 못 된 씨앗: {'B0 계좌 공유': 917, 'B1 계좌 공유 + 7일 이내': 927, 'B2 계좌 공유 + 1일 이내': 1735, 'B 4일 이내': 996}


## 4. 정답과 비교

In [8]:
summaries, per_attempt = [], {}
for name, groups in methods.items():
    summary, per = GL.compare_to_truth(name, groups, attempt_of, eval_attempts, truth=truth)
    summaries.append(summary)
    per_attempt[name] = per
result = pd.DataFrame(summaries).set_index('방법').T.astype(object)   # 정수는 정수로 보이게
result

방법,G 생성기,B0 계좌 공유,B1 계좌 공유 + 7일 이내,B2 계좌 공유 + 1일 이내,B 4일 이내
묶음_수,1259.0,1008.0,1011.0,1651.0,1040.0
묶인_거래,9668.0,10110.0,10100.0,9292.0,10031.0
크기_중앙값,5.0,7.0,7.0,4.0,6.0
크기_최대,62.0,176.0,176.0,176.0,176.0
세탁_비율,0.9496,0.8974,0.8983,0.9099,0.9023
전부_정상_묶음,51.0,33.0,32.0,22.0,31.0
시도_2개이상_섞인_묶음,97.0,121.0,119.0,129.0,111.0
쌍_정밀도,0.6571,0.3384,0.3399,0.3211,0.3536
쌍_재현율,0.8832,0.972,0.972,0.5593,0.9691
시도_수,1291.0,1291.0,1291.0,1291.0,1291.0


**읽는 법**
- `세탁_비율`: 묶인 거래 중 세탁 거래의 비율.
- `시도_2개이상_섞인_묶음`: 서로 다른 세탁 시도가 한 묶음에 들어간 수. 조사자가 한 사건으로 보게 되는 것이 실제로는 여러 사건이다.
- `쌍_정밀도`: 같은 묶음에 든 두 거래가 실제로 같은 시도일 비율. 큰 묶음에 이것저것 섞이면 급격히 떨어진다.
- `쌍_재현율`: 같은 시도의 두 거래가 같은 묶음에 들어갔을 비율.
- `온전히_한_묶음`: 시도의 (구간 안) 거래 전부가 한 묶음에 들어간 시도 수. 2절의 `모델이_전부_잡은_시도` 가 사실상의 상한이다.
- `평균_최대_포함률`: 시도마다 가장 많이 담은 묶음 하나가 그 시도 거래의 몇 %를 담았는지의 평균. 큰 시도는 이 값으로 본다.
- `쪼개진_시도`: 시도의 거래가 묶음 2개 이상에 나뉜 수.

## 5. 패턴별

In [9]:
frames = []
for name, per in per_attempt.items():
    f = per.groupby('pattern_type').agg(시도=('size', 'size'), 온전=('coverage', lambda s: round(float((s == 1).mean()), 3)),
                                        포함률=('coverage', lambda s: round(float(s.mean()), 3)),
                                        쪼개짐=('n_groups', lambda s: round(float((s >= 2).mean()), 3)))
    frames.append(f.add_prefix(name.split()[0] + '_'))
by_pattern = pd.concat(frames, axis=1)
by_pattern

,G_시도,G_온전,G_포함률,G_쪼개짐,B0_시도,B0_온전,B0_포함률,B0_쪼개짐,B1_시도,B1_온전,B1_포함률,B1_쪼개짐,B2_시도,B2_온전,B2_포함률,B2_쪼개짐,B_시도,B_온전,B_포함률,B_쪼개짐
pattern_type,,,,,,,,,,,,,,,,,,,,
CYCLE,236,0.758,0.849,0.127,236,0.949,0.972,0.017,236,0.949,0.972,0.017,236,0.403,0.699,0.390,236,0.924,0.953,0.030
FAN-IN,209,0.756,0.905,0.057,209,0.833,0.949,0.000,209,0.833,0.949,0.000,209,0.349,0.710,0.301,209,0.823,0.942,0.005
FAN-OUT,183,0.825,0.932,0.071,183,0.945,0.983,0.000,183,0.945,0.983,0.000,183,0.508,0.783,0.257,183,0.929,0.974,0.005
GATHER-SCATTER,267,0.476,0.789,0.199,267,0.861,0.977,0.000,267,0.861,0.977,0.000,267,0.363,0.749,0.476,267,0.854,0.976,0.004
RANDOM,168,0.833,0.910,0.107,168,0.964,0.981,0.018,168,0.964,0.981,0.018,168,0.327,0.655,0.393,168,0.952,0.979,0.018
SCATTER-GATHER,228,0.803,0.913,0.057,228,0.921,0.986,0.000,228,0.921,0.986,0.000,228,0.390,0.734,0.487,228,0.917,0.985,0.004


## 5-1. 시도 크기별
거래 2건짜리 시도와 큰 시도는 묶기 난이도가 다르다. 크기(평가 구간 안 거래 수)로 나눠 본다.

In [10]:
SIZE_BINS, SIZE_LABELS = [1, 2, 5, 10, 20, 10_000], ['2건', '3~5건', '6~10건', '11~20건', '21건 이상']
frames = []
for name, per in per_attempt.items():
    bucket = pd.cut(per['size'], SIZE_BINS, labels=SIZE_LABELS)
    f = per.groupby(bucket, observed=True).agg(시도=('size', 'size'), 온전=('coverage', lambda s: round(float((s == 1).mean()), 3)),
                                               포함률=('coverage', lambda s: round(float(s.mean()), 3)),
                                               쪼개짐=('n_groups', lambda s: round(float((s >= 2).mean()), 3)))
    frames.append(f.add_prefix(name.split()[0] + '_'))
by_size = pd.concat(frames, axis=1)
by_size

,G_시도,G_온전,G_포함률,G_쪼개짐,B0_시도,B0_온전,B0_포함률,B0_쪼개짐,B1_시도,B1_온전,B1_포함률,B1_쪼개짐,B2_시도,B2_온전,B2_포함률,B2_쪼개짐,B_시도,B_온전,B_포함률,B_쪼개짐
size,,,,,,,,,,,,,,,,,,,,
2건,291,0.660,0.744,0.069,291,0.948,0.957,0.000,291,0.948,0.957,0.000,291,0.656,0.737,0.107,291,0.928,0.938,0.003
3~5건,371,0.782,0.908,0.081,371,0.938,0.976,0.008,371,0.938,0.976,0.008,371,0.464,0.759,0.183,371,0.927,0.972,0.013
6~10건,376,0.750,0.921,0.122,376,0.899,0.980,0.011,376,0.899,0.980,0.011,376,0.271,0.716,0.561,376,0.888,0.976,0.019
11~20건,218,0.674,0.914,0.183,218,0.839,0.986,0.000,218,0.839,0.986,0.000,218,0.151,0.679,0.757,218,0.830,0.983,0.005
21건 이상,35,0.771,0.944,0.086,35,0.800,0.991,0.000,35,0.800,0.991,0.000,35,0.114,0.596,0.886,35,0.800,0.991,0.000


## 5-2. 평가 구간 뒤로 이어지는 시도 vs 구간 안에 끝난 시도
평가 구간이 끝난 뒤에도 거래가 이어지는 시도는 구간 안에서 **완성되기 전 모습**만 보인다. 모델도 묶기도 불리하다.
- `구간 안에 끝남`: 그 시도의 마지막 거래가 평가 구간 마지막 날 이전
- `구간 뒤로 이어짐`: 마지막 거래가 그 뒤

`모델이_전부_잡음` 은 묶기 전 단계의 상한이다. 묶기 방법의 `온전` 은 이 값을 넘기 어렵다.
(시작이 평가 구간 앞인 시도는 따로 나누지 않았다.)

In [11]:
after = attempts.set_index('attempt_id').last_ts >= end          # 시도 → 구간 뒤로 이어지는가
ends = pd.Series(by_attempt.index.map(after).map(GL.TAIL_LABELS), index=by_attempt.index, name='끝')
ceiling_tail = (by_attempt.assign(전부=by_attempt.seeds == by_attempt['size']).groupby(ends)
                .agg(시도=('size', 'size'), 구간_안_거래=('size', 'sum'), 모델이_잡은_거래=('seeds', 'sum'), 모델이_전부_잡음=('전부', 'mean')).round(3))
ceiling_tail['거래_재현율'] = (ceiling_tail.모델이_잡은_거래 / ceiling_tail.구간_안_거래).round(3)
ceiling_tail

,시도,구간_안_거래,모델이_잡은_거래,모델이_전부_잡음,거래_재현율
끝,,,,,
구간 뒤로 이어짐,451,3236,3169,0.887,0.979
구간 안에 끝남,840,5661,5598,0.937,0.989


In [12]:
def tail_table(keys):
    frames = []
    for name, per in per_attempt.items():
        per = per.assign(끝=per.attempt_id.map(after).map(GL.TAIL_LABELS))
        f = per.groupby(keys).agg(시도=('size', 'size'), 온전=('coverage', lambda s: round(float((s == 1).mean()), 3)),
                                  포함률=('coverage', lambda s: round(float(s.mean()), 3)),
                                  쪼개짐=('n_groups', lambda s: round(float((s >= 2).mean()), 3)))
        frames.append(f.add_prefix(name.split()[0] + '_'))
    return pd.concat(frames, axis=1)

by_tail = tail_table('끝')
by_tail

,G_시도,G_온전,G_포함률,G_쪼개짐,B0_시도,B0_온전,B0_포함률,B0_쪼개짐,B1_시도,B1_온전,B1_포함률,B1_쪼개짐,B2_시도,B2_온전,B2_포함률,B2_쪼개짐,B_시도,B_온전,B_포함률,B_쪼개짐
끝,,,,,,,,,,,,,,,,,,,,
구간 뒤로 이어짐,451,0.716,0.893,0.120,451,0.869,0.962,0.009,451,0.869,0.962,0.009,451,0.377,0.736,0.379,451,0.863,0.959,0.013
구간 안에 끝남,840,0.732,0.868,0.101,840,0.930,0.981,0.004,840,0.930,0.981,0.004,840,0.395,0.717,0.399,840,0.914,0.973,0.010


In [13]:
# 패턴 × 끝
by_pattern_tail = tail_table(['pattern_type', '끝'])
by_pattern_tail

G_시도   G_온전  G_포함률  G_쪼개짐  B0_시도  B0_온전  B0_포함률  \
pattern_type   끝                                                            
CYCLE          구간 뒤로 이어짐    69  0.812  0.885  0.087     69  0.942   0.967   
               구간 안에 끝남    167  0.737  0.835  0.144    167  0.952   0.975   
FAN-IN         구간 뒤로 이어짐    70  0.757  0.883  0.029     70  0.843   0.933   
               구간 안에 끝남    139  0.755  0.916  0.072    139  0.827   0.957   
FAN-OUT        구간 뒤로 이어짐    57  0.825  0.920  0.070     57  0.912   0.958   
               구간 안에 끝남    126  0.825  0.937  0.071    126  0.960   0.994   
GATHER-SCATTER 구간 뒤로 이어짐   145  0.510  0.872  0.234    145  0.793   0.968   
               구간 안에 끝남    122  0.434  0.691  0.156    122  0.943   0.988   
RANDOM         구간 뒤로 이어짐    45  0.867  0.892  0.044     45  0.911   0.943   
               구간 안에 끝남    123  0.821  0.917  0.130    123  0.984   0.995   
SCATTER-GATHER 구간 뒤로 이어짐    65  0.831  0.938  0.092     65  0.923   0.991   
               구간 안에 끝남    163  0.791  0.903  0.043    163  0.920   0.984   

                          B0_쪼개짐  B1_시도  B1_온전  B1_포함률  B1_쪼개짐  B2_시도  B2_온전  \
pattern_type   끝                                                               
CYCLE          구간 뒤로 이어짐   0.029     69  0.942   0.967   0.029     69  0.464   
               구간 안에 끝남    0.012    167  0.952   0.975   0.012    167  0.377   
FAN-IN         구간 뒤로 이어짐   0.000     70  0.843   0.933   0.000     70  0.371   
               구간 안에 끝남    0.000    139  0.827   0.957   0.000    139  0.338   
FAN-OUT        구간 뒤로 이어짐   0.000     57  0.912   0.958   0.000     57  0.526   
               구간 안에 끝남    0.000    126  0.960   0.994   0.000    126  0.500   
GATHER-SCATTER 구간 뒤로 이어짐   0.000    145  0.793   0.968   0.000    145  0.241   
               구간 안에 끝남    0.000    122  0.943   0.988   0.000    122  0.508   
RANDOM         구간 뒤로 이어짐   0.044     45  0.911   0.943   0.044     45  0.422   
               구간 안에 끝남    0.008    123  0.984   0.995   0.008    123  0.293   
SCATTER-GATHER 구간 뒤로 이어짐   0.000     65  0.923   0.991   0.000     65  0.431   
               구간 안에 끝남    0.000    163  0.920   0.984   0.000    163  0.374   

                          B2_포함률  B2_쪼개짐  B_시도   B_온전  B_포함률  B_쪼개짐  
pattern_type   끝                                                     
CYCLE          구간 뒤로 이어짐   0.756   0.275    69  0.942  0.967  0.029  
               구간 안에 끝남    0.675   0.437   167  0.916  0.948  0.030  
FAN-IN         구간 뒤로 이어짐   0.722   0.243    70  0.829  0.919  0.000  
               구간 안에 끝남    0.705   0.331   139  0.820  0.954  0.007  
FAN-OUT        구간 뒤로 이어짐   0.806   0.246    57  0.912  0.958  0.000  
               구간 안에 끝남    0.773   0.262   126  0.937  0.981  0.008  
GATHER-SCATTER 구간 뒤로 이어짐   0.694   0.572   145  0.786  0.966  0.007  
               구간 안에 끝남    0.815   0.361   122  0.934  0.986  0.000  
RANDOM         구간 뒤로 이어짐   0.715   0.222    45  0.911  0.943  0.044  
               구간 안에 끝남    0.634   0.455   123  0.967  0.992  0.008  
SCATTER-GATHER 구간 뒤로 이어짐   0.776   0.431    65  0.908  0.988  0.015  
               구간 안에 끝남    0.717   0.509   163  0.920  0.984  0.000

## 6. 큰 묶음 들여다보기
단순 연결은 계좌 하나가 여러 사건을 이어 버리면 묶음이 커진다. 방법마다 가장 큰 묶음 5개를 본다.

In [14]:
rows = []
for name, groups in methods.items():
    g = groups.merge(attempt_of[['edge_id', 'attempt_id']], on='edge_id', how='left')
    stat = g.groupby('group').agg(거래=('edge_id', 'size'), 세탁=('y', 'sum'), 시도_수=('attempt_id', 'nunique'))
    for group, r in stat.sort_values('거래', ascending=False).head(5).iterrows():
        rows.append(dict(방법=name, 거래=int(r.거래), 세탁=int(r.세탁), 시도_수=int(r.시도_수)))
pd.DataFrame(rows)

,방법,거래,세탁,시도_수
0,G 생성기,62,62,10
1,G 생성기,62,62,15
2,G 생성기,61,60,13
3,G 생성기,57,56,19
4,G 생성기,54,54,13
5,B0 계좌 공유,176,165,20
6,B0 계좌 공유,118,108,16
7,B0 계좌 공유,115,111,18
8,B0 계좌 공유,108,102,12
9,B0 계좌 공유,104,100,19


## 7. 저장

In [15]:
result.to_csv(OUT / 'summary.csv')
by_pattern.to_csv(OUT / 'by_pattern.csv')
by_size.to_csv(OUT / 'by_size.csv')
ceiling_tail.to_csv(OUT / 'ceiling_tail.csv'); by_tail.to_csv(OUT / 'by_tail.csv'); by_pattern_tail.to_csv(OUT / 'by_pattern_tail.csv')
pd.concat([per.assign(method=name) for name, per in per_attempt.items()]).to_csv(OUT / 'per_attempt.csv', index=False)
json.dump(dict(run=RUN, exclude=EXCLUDE, baselines=BASELINES, ceiling=ceiling, lonely_seeds=lonely),
          open(OUT / 'run.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print(sorted(p.name for p in OUT.iterdir()))

['by_pattern.csv', 'by_pattern_tail.csv', 'by_size.csv', 'by_tail.csv', 'ceiling_tail.csv', 'per_attempt.csv', 'run.json', 'summary.csv']
